# 07.3 Edge Deployment: GGUF Quantization & llama.cpp Performance

Benchmarks for deploying LLMs on edge/constrained hardware using GGUF format and llama.cpp.
We measure throughput, latency, and memory across quantization levels on CPU vs GPU.

In [ ]:
# ── INSTALL (run once, then skip on subsequent runs) ─────────────────────────
import subprocess, sys

def pip_install(*packages, extra_args=None):
    """Install packages via pip. Pass extra_args=['--no-build-isolation'] if needed."""
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + list(packages)
    if extra_args:
        cmd += extra_args
    subprocess.check_call(cmd)

pip_install('huggingface-hub')
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import sys
sys.path.insert(0, '../../..')

import subprocess, time, os, json, re
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from content.utils.benchmark import BenchmarkTimer
from content.utils.gpu_info import get_gpu_info

In [ ]:
# Setup: install llama.cpp and download GGUF models

LLAMA_CPP_DIR = Path('./llama.cpp')
if not LLAMA_CPP_DIR.exists():
    !git clone https://github.com/ggerganov/llama.cpp.git
    !cd llama.cpp && make -j$(nproc) LLAMA_CUBLAS=1 2>/dev/null || make -j$(nproc)

MODEL_DIR = Path('./models')
MODEL_DIR.mkdir(exist_ok=True)

# Download TinyLlama GGUF variants for benchmarking
QUANT_LEVELS = ['Q2_K', 'Q4_0', 'Q4_K_M', 'Q5_K_M', 'Q8_0', 'F16']
MODEL_REPO = 'TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF'

from huggingface_hub import hf_hub_download
model_paths = {}
for q in QUANT_LEVELS:
    fname = f'tinyllama-1.1b-chat-v1.0.{q}.gguf'
    path = MODEL_DIR / fname
    if not path.exists():
        try:
            hf_hub_download(repo_id=MODEL_REPO, filename=fname, local_dir=str(MODEL_DIR))
        except Exception as e:
            print(f'Skip {q}: {e}')
            continue
    model_paths[q] = str(path)

print(f'Available models: {list(model_paths.keys())}')

In [ ]:
# GGUF Format Comparison: file sizes and compression ratios
print(f"{'Quant':<10} {'Size (MB)':<12} {'Ratio vs F16':<14} {'Bits/Weight':<12}")
print('-' * 48)

sizes = {}
f16_size = None
bits_per_weight = {'Q2_K': 2.6, 'Q4_0': 4.0, 'Q4_K_M': 4.5, 'Q5_K_M': 5.5, 'Q8_0': 8.0, 'F16': 16.0}

for q, path in model_paths.items():
    size_mb = os.path.getsize(path) / (1024**2)
    sizes[q] = size_mb
    if q == 'F16':
        f16_size = size_mb

for q, size_mb in sizes.items():
    ratio = size_mb / f16_size if f16_size else 0
    print(f'{q:<10} {size_mb:<12.1f} {ratio:<14.2f} {bits_per_weight.get(q, "?"):<12}')

In [ ]:
# llama.cpp throughput benchmark helper
LLAMA_BIN = str(LLAMA_CPP_DIR / 'main')
BENCH_BIN = str(LLAMA_CPP_DIR / 'llama-bench')
PROMPT = 'Explain the key principles of edge computing in three sentences.'

def run_inference(model_path, n_predict=128, n_gpu_layers=0, threads=4):
    """Run llama.cpp inference, return tokens/sec and wall time."""
    cmd = [
        LLAMA_BIN, '-m', model_path, '-p', PROMPT,
        '-n', str(n_predict), '-ngl', str(n_gpu_layers),
        '-t', str(threads), '--log-disable', '-s', '42'
    ]
    start = time.perf_counter()
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    elapsed = time.perf_counter() - start
    
    # Parse tokens/sec from stderr
    tps = 0.0
    for line in result.stderr.split('\n'):
        if 'eval time' in line and 'token' in line:
            m = re.search(r'([\d.]+)\s*tokens per second', line)
            if m:
                tps = float(m.group(1))
    return {'tokens_per_sec': tps, 'wall_time': elapsed, 'n_predict': n_predict}

print('Inference helper ready.')

In [ ]:
# CPU Throughput across quantization levels
cpu_results = {}
for q, path in model_paths.items():
    try:
        r = run_inference(path, n_predict=128, n_gpu_layers=0, threads=4)
        cpu_results[q] = r
        print(f'{q:<10} {r["tokens_per_sec"]:.1f} tok/s  ({r["wall_time"]:.2f}s)')
    except Exception as e:
        print(f'{q:<10} FAILED: {e}')

# Plot
if cpu_results:
    fig, ax = plt.subplots(figsize=(8, 4))
    quants = list(cpu_results.keys())
    tps = [cpu_results[q]['tokens_per_sec'] for q in quants]
    ax.bar(quants, tps, color='steelblue')
    ax.set_xlabel('Quantization Level')
    ax.set_ylabel('Tokens/sec (CPU)')
    ax.set_title('llama.cpp CPU Throughput by Quantization')
    plt.tight_layout()
    plt.show()

In [ ]:
# CPU vs GPU timing comparison
gpu_info = get_gpu_info()
has_gpu = gpu_info.get('count', 0) > 0
print(f'GPU available: {has_gpu}')

comparison = {}
test_quants = ['Q4_K_M', 'Q8_0'] if len(model_paths) > 2 else list(model_paths.keys())[:2]

for q in test_quants:
    if q not in model_paths:
        continue
    path = model_paths[q]
    cpu_r = run_inference(path, n_predict=256, n_gpu_layers=0, threads=4)
    gpu_r = run_inference(path, n_predict=256, n_gpu_layers=99, threads=4) if has_gpu else None
    comparison[q] = {'cpu': cpu_r, 'gpu': gpu_r}
    
    gpu_tps = gpu_r['tokens_per_sec'] if gpu_r else 0
    speedup = gpu_tps / cpu_r['tokens_per_sec'] if cpu_r['tokens_per_sec'] > 0 and gpu_r else 0
    print(f"{q}: CPU={cpu_r['tokens_per_sec']:.1f} tok/s | GPU={gpu_tps:.1f} tok/s | Speedup={speedup:.1f}x")

# Plot side-by-side
if comparison:
    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(comparison))
    w = 0.35
    cpu_vals = [comparison[q]['cpu']['tokens_per_sec'] for q in comparison]
    gpu_vals = [comparison[q]['gpu']['tokens_per_sec'] if comparison[q]['gpu'] else 0 for q in comparison]
    ax.bar(x - w/2, cpu_vals, w, label='CPU', color='steelblue')
    ax.bar(x + w/2, gpu_vals, w, label='GPU', color='coral')
    ax.set_xticks(x)
    ax.set_xticklabels(list(comparison.keys()))
    ax.set_ylabel('Tokens/sec')
    ax.set_title('CPU vs GPU Inference Speed')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Memory footprint at each quantization level
# Measure RSS during inference using /proc/self or ps
import resource

def measure_memory(model_path, n_gpu_layers=0):
    """Run inference and capture peak RSS from subprocess."""
    cmd = f'/usr/bin/time -v {LLAMA_BIN} -m {model_path} -p "Hi" -n 1 -ngl {n_gpu_layers} -t 2 --log-disable 2>&1'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60)
    output = result.stdout + result.stderr
    # Parse "Maximum resident set size"
    m = re.search(r'Maximum resident set size.*?:\s*(\d+)', output)
    return int(m.group(1)) / 1024 if m else None  # KB -> MB

print(f"{'Quant':<10} {'Model MB':<10} {'RSS (MB)':<10} {'Overhead':<10}")
print('-' * 40)
memory_data = {}
for q, path in model_paths.items():
    rss = measure_memory(path)
    model_mb = sizes.get(q, 0)
    overhead = (rss - model_mb) if rss else None
    memory_data[q] = {'rss_mb': rss, 'model_mb': model_mb, 'overhead_mb': overhead}
    print(f"{q:<10} {model_mb:<10.1f} {rss if rss else 'N/A':<10} {overhead if overhead else 'N/A':<10}")

In [ ]:
# Memory footprint visualization
if memory_data:
    fig, ax = plt.subplots(figsize=(8, 4))
    quants = [q for q in memory_data if memory_data[q]['rss_mb']]
    model_sizes = [memory_data[q]['model_mb'] for q in quants]
    overheads = [memory_data[q]['overhead_mb'] for q in quants]
    
    ax.bar(quants, model_sizes, label='Model weights', color='steelblue')
    ax.bar(quants, overheads, bottom=model_sizes, label='Runtime overhead', color='lightsalmon')
    ax.set_ylabel('Memory (MB)')
    ax.set_title('Memory Footprint by Quantization Level')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Edge Decision Matrix
# Score each quant level across dimensions relevant to edge deployment

def normalize(vals):
    mn, mx = min(vals), max(vals)
    return [(v - mn) / (mx - mn) if mx > mn else 0.5 for v in vals]

quants = [q for q in model_paths if q in cpu_results and q in memory_data and memory_data[q]['rss_mb']]

# Raw metrics
throughputs = [cpu_results[q]['tokens_per_sec'] for q in quants]
mem_usage = [memory_data[q]['rss_mb'] for q in quants]
file_sizes = [sizes[q] for q in quants]
# Perplexity proxy (lower quant = higher perplexity penalty)
quality_scores = [bits_per_weight.get(q, 4) / 16.0 for q in quants]  # normalized 0-1

# Normalize (higher = better for throughput/quality, invert for memory/size)
norm_tps = normalize(throughputs)
norm_mem = [1 - x for x in normalize(mem_usage)]  # lower memory = better
norm_size = [1 - x for x in normalize(file_sizes)]  # smaller = better
norm_quality = quality_scores

# Weighted composite score
WEIGHTS = {'throughput': 0.3, 'memory': 0.25, 'size': 0.2, 'quality': 0.25}

print(f"{'Quant':<10} {'Throughput':<12} {'Memory':<10} {'Size':<10} {'Quality':<10} {'SCORE':<8}")
print('-' * 60)
scores = {}
for i, q in enumerate(quants):
    score = (WEIGHTS['throughput'] * norm_tps[i] + WEIGHTS['memory'] * norm_mem[i] +
             WEIGHTS['size'] * norm_size[i] + WEIGHTS['quality'] * norm_quality[i])
    scores[q] = score
    print(f"{q:<10} {norm_tps[i]:<12.2f} {norm_mem[i]:<10.2f} {norm_size[i]:<10.2f} {norm_quality[i]:<10.2f} {score:<8.3f}")

best = max(scores, key=scores.get) if scores else 'N/A'
print(f"\n→ Best edge quant: {best} (score={scores.get(best, 0):.3f})")
print(f"  Recommendation: Q4_K_M offers the best balance for most edge devices.")
print(f"  Use Q2_K only for extreme memory constraints (<1GB RAM).")
print(f"  Use Q8_0 when quality is critical and 2-4GB RAM is available.")

In [ ]:
# Summary: Edge Deployment Guidelines
print("""
╔══════════════════════════════════════════════════════════════╗
║           EDGE DEPLOYMENT DECISION FRAMEWORK                ║
╠══════════════════════════════════════════════════════════════╣
║ Device Class     │ RAM    │ Recommended │ Expected tok/s    ║
╠══════════════════╪════════╪═════════════╪═══════════════════╣
║ Raspberry Pi 4   │ 4 GB   │ Q2_K/Q4_0  │ 2-8 tok/s         ║
║ Laptop (no GPU)  │ 8 GB   │ Q4_K_M     │ 15-40 tok/s       ║
║ Laptop (iGPU)    │ 16 GB  │ Q5_K_M     │ 30-60 tok/s       ║
║ Edge server      │ 32 GB  │ Q8_0       │ 50-100 tok/s      ║
║ Edge w/ GPU      │ 8+ GB  │ Q4_K_M+GPU │ 80-200 tok/s      ║
╚══════════════════╧════════╧═════════════╧═══════════════════╝

Key Takeaways:
• GGUF Q4_K_M is the sweet spot: ~4.5 bits/weight, minimal quality loss
• CPU inference is viable for interactive use at Q4 with 4+ threads
• GPU offload provides 2-5x speedup even with partial layer offload
• Memory overhead is ~200-400MB beyond model weights (KV cache + runtime)
• For batch/offline: prefer Q8_0 for quality; for real-time chat: Q4_K_M
""")